### DATA LOADING MODULE
- Imports

In [15]:

import os
import time
import json
import warnings
import requests
import pandas as pd
import numpy as np
from datetime import timedelta, datetime
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
load_dotenv()

True

- CONFIGURATION 

In [16]:
DUNE_API_KEYS = [
    os.getenv("DUNE_LASEVEN7"),
    os.getenv("DUNE_FIRSTBML"),
    os.getenv("DUNE_LASEVEN71"),
    os.getenv("DUNE_LASEVEN7_TEAM"),
    os.getenv("DUNE_FIRSTBML_TEAM")
]

DUNE_API_KEYS = [key for key in DUNE_API_KEYS if key and str(key).strip()]
print("🔍 Loaded API Keys:")
env_names = [
    "DUNE_LASEVEN7", 
    "DUNE_FIRSTBML",
    "DUNE_LASEVEN71",
    "DUNE_LASEVEN7_TEAM",
    "DUNE_FIRSTBML_TEAM"
]
for i, key in enumerate(DUNE_API_KEYS):
    env_name = env_names[i] if i < len(env_names) else "UNKNOWN"
    print(f"   ✅ {i+1}. {env_name}: {key[:8]}...")

if not DUNE_API_KEYS:
    raise ValueError("No Dune API keys found!")

COINGECKO_API_KEY = os.getenv("COINGECKO_API_KEY")

for d in ['data', 'data/price_cache', 'logs']:
    os.makedirs(d, exist_ok=True)

QUERIES = {
    "whales": ("6395391", "data/dune_whales_cache.json", "data/whale_ml_ready.csv"),
    "market_intent": ("6385600", "data/dune_intent_cache.json", "data/market_intent_ml_ready.csv")
}

DUNE_START = pd.Timestamp('2017-10-16', tz='UTC')
COINGECKO_BTC_START = '01-01-2013'
COINGECKO_ETH_START = '01-08-2015'


🔍 Loaded API Keys:
   ✅ 1. DUNE_LASEVEN7: D1Smlg20...
   ✅ 2. DUNE_FIRSTBML: fHg07khx...
   ✅ 3. DUNE_LASEVEN71: hueVX9VB...
   ✅ 4. DUNE_LASEVEN7_TEAM: 7ND5DKw8...
   ✅ 5. DUNE_FIRSTBML_TEAM: p2UxSBFJ...


- KEY ROTATION

In [17]:

class DuneKeyRotator:
    def __init__(self, api_keys):
        self.api_keys = api_keys
        self.key_index = 0
        self.key_usage = {key: {"count": 0, "last_used": None, "errors": 0, "exhausted": False} for key in api_keys}
        self.total_requests = 0
    
    def get_next_key(self):
        if not self.api_keys:
            raise ValueError("No API keys available")
        
        available_keys = []
        for key in self.api_keys:
            usage = self.key_usage[key]
            
            if usage["exhausted"]:  # Skip exhausted keys
                continue
                
            if usage["errors"] >= 3:
                continue
                
            if usage["last_used"]:
                time_since_use = (datetime.now() - usage["last_used"]).total_seconds()
                if time_since_use < 60:
                    continue
                    
            available_keys.append((key, usage["count"], usage["errors"]))
        
        if not available_keys:
            # Check if ALL keys are exhausted
            if all(usage["exhausted"] for usage in self.key_usage.values()):
                raise Exception("ALL_API_KEYS_EXHAUSTED")
            
            available_keys = [(key, self.key_usage[key]["count"], self.key_usage[key]["errors"]) 
                            for key in self.api_keys if not self.key_usage[key]["exhausted"]]
        
        if not available_keys:
            raise Exception("NO_KEYS_AVAILABLE")
        
        available_keys.sort(key=lambda x: (x[2], x[1]))
        selected_key = available_keys[0][0]
        
        self.key_usage[selected_key]["count"] += 1
        self.key_usage[selected_key]["last_used"] = datetime.now()
        self.total_requests += 1
        
        return selected_key
    
    def mark_exhausted(self, key):
        """Mark a key as exhausted (reached monthly limit)"""
        if key in self.key_usage:
            self.key_usage[key]["exhausted"] = True
            print(f"⛔ KEY {key[:8]}... MARKED AS EXHAUSTED (monthly limit reached)")
    
    def are_all_keys_exhausted(self):
        """Check if all keys are exhausted"""
        return all(usage["exhausted"] for usage in self.key_usage.values())
        
    def mark_error(self, key):
        if key in self.key_usage:
            self.key_usage[key]["errors"] += 1
    
    def mark_success(self, key):
        if key in self.key_usage and self.key_usage[key]["errors"] > 0:
            self.key_usage[key]["errors"] = max(0, self.key_usage[key]["errors"] - 1)
    
    def get_stats(self):
        return {
            "total_requests": self.total_requests,
            "active_keys": sum(1 for key, stats in self.key_usage.items() if stats["errors"] < 5)
        }

key_rotator = DuneKeyRotator(DUNE_API_KEYS)

- API CLIENT

In [18]:

class DuneAPIClient:
    def __init__(self, key_rotator):
        self.key_rotator = key_rotator
        self.session = requests.Session()
        self.session.headers.update({
            "Content-Type": "application/json",
            "Accept": "application/json"
        })
    
    def make_request(self, method, url, **kwargs):
        max_retries = len(self.key_rotator.api_keys)  # Try all keys
        
        for attempt in range(max_retries):
            try:
                api_key = self.key_rotator.get_next_key()
                headers = kwargs.get('headers', {}).copy()
                headers["x-dune-api-key"] = api_key
                kwargs['headers'] = headers
                
                response = self.session.request(method, url, **kwargs)
                
                if response.status_code == 402:
                    # This key is exhausted for the month
                    self.key_rotator.mark_exhausted(api_key)
                    self.key_rotator.mark_error(api_key)
                    
                    # Check if all keys are now exhausted
                    if self.key_rotator.are_all_keys_exhausted():
                        return response  # Return 402 to signal complete exhaustion
                    
                    continue  # Try another key
                    
                elif response.status_code == 200:
                    self.key_rotator.mark_success(api_key)
                    return response
                    
                else:
                    self.key_rotator.mark_error(api_key)
                    
                    if attempt == max_retries - 1:
                        return response
                        
                    time.sleep(2)
                    
            except Exception as e:
                self.key_rotator.mark_error(api_key)
                
                if attempt == max_retries - 1:
                    raise
        
        raise Exception(f"Failed after {max_retries} attempts")
    
    def get(self, url, **kwargs):
        return self.make_request("GET", url, **kwargs)
    
    def post(self, url, **kwargs):
        return self.make_request("POST", url, **kwargs)

dune_client = DuneAPIClient(key_rotator)



- UTILITY FUNCTIONS
 - Dune

In [19]:

def to_utc(ts):
    ts = pd.Timestamp(ts)
    return ts.tz_localize("UTC") if ts.tzinfo is None else ts.tz_convert("UTC")


def fetch_dune_incremental(qid, cache_path, query_name="whale_data", force_fetch=False):
    """
    Stops on rate limit instead of skipping dates
    """
    today = pd.Timestamp.now(tz='UTC').normalize()
    yesterday = today - pd.Timedelta(days=1)
    
    print(f"\n📊 Fetching {query_name}...")
    print(f"   Using {len(DUNE_API_KEYS)} API keys")
    
    df_cached = pd.DataFrame()
    last_date = None
    
    if os.path.exists(cache_path) and not force_fetch:
        try:
            with open(cache_path) as f:
                cached = json.load(f)
            
            if 'data' in cached and cached['data']:
                df_cached = pd.DataFrame(cached["data"])
                
                if 'block_date' in df_cached.columns:
                    df_cached["block_date"] = pd.to_datetime(df_cached["block_date"], utc=True)
                    
                    if 'is_estimate' in df_cached.columns:
                        df_cached = df_cached.drop('is_estimate', axis=1)
                    
                    last_date = df_cached["block_date"].max()
                    
                    if last_date >= yesterday:
                        print(f"✅ {query_name} cache current ({last_date.date()})")
                        return df_cached
                    
                    print(f"📅 Cache: {last_date.date()}, fetching new data...")
                else:
                    last_date = DUNE_START
            else:
                last_date = DUNE_START
                
        except Exception as e:
            print(f"⚠️  Cache error: {e}")
            last_date = DUNE_START
    else:
        print(f"📝 Fetching from {DUNE_START.date()}...")
        last_date = DUNE_START
    
    fetch_start = last_date + pd.Timedelta(days=1) if last_date else DUNE_START
    fetch_end = yesterday
    
    if fetch_start > fetch_end:
        return df_cached
    
    print(f"🔍 Fetching: {fetch_start.date()} to {fetch_end.date()}")
    
    # FETCH ONE DAY AT A TIME
    all_new_rows = []
    current_date = fetch_start
    
    while current_date <= fetch_end:
        query_params = {
            "start_date": current_date.strftime("%Y-%m-%d"),
            "end_date": current_date.strftime("%Y-%m-%d")
        }
        
        try:
            execute_url = f"https://api.dune.com/api/v1/query/{qid}/execute"
            execute_payload = {"query_parameters": query_params}
            
            print(f"   🔸 {current_date.date()}...", end="")
            
            resp = dune_client.post(execute_url, json=execute_payload, timeout=60)
            
            # ✅ Stop on 402, don't skip
            if resp.status_code != 200:
                if resp.status_code == 402:
                    print(f" ❌ 402 RATE LIMIT")
                    print(f"\n⚠️  STOPPED at {current_date.date()} - rate limit hit")
                    print(f"   Cache saved up to: {last_date.date() if last_date else 'N/A'}")
                    print(f"   Resume tomorrow when limits reset")
                    break  # STOP completely, don't skip
                
                print(f" ❌ {resp.status_code}")
                current_date += pd.Timedelta(days=1)
                time.sleep(2)
                continue
            
            resp_json = resp.json()
            
            if 'execution_id' not in resp_json:
                print(f" ❌ No execution_id")
                current_date += pd.Timedelta(days=1)
                time.sleep(2)
                continue
            
            eid = resp_json["execution_id"]
            
            # Wait for completion
            for attempt in range(60):
                status_url = f"https://api.dune.com/api/v1/execution/{eid}/status"
                
                try:
                    status_resp = dune_client.get(status_url, timeout=30)
                    
                    if status_resp.status_code == 200:
                        status_data = status_resp.json()
                        state = status_data.get("state", "UNKNOWN")
                        
                        if state == "QUERY_STATE_COMPLETED":
                            break
                        elif state in ["QUERY_STATE_FAILED", "QUERY_STATE_CANCELLED"]:
                            print(f" ❌ {state}")
                            break
                    
                except Exception as e:
                    print(f" ⚠️ {e}")
                    break
                
                time.sleep(5)
            else:
                print(f" ⏱️ Timeout")
                current_date += pd.Timedelta(days=1)
                time.sleep(2)
                continue
            
            # Get results
            results_url = f"https://api.dune.com/api/v1/execution/{eid}/results"
            results_resp = dune_client.get(results_url, timeout=30)
            
            if results_resp.status_code == 200:
                results_data = results_resp.json()
                
                if 'result' in results_data and 'rows' in results_data['result']:
                    rows = results_data["result"]["rows"]
                    if rows:
                        all_new_rows.extend(rows)
                        print(f" ✅ {len(rows)} rows")
                    else:
                        print(f" ⚠️ No data")
                else:
                    print(f" ❌ No results")
            else:
                print(f" ❌ {results_resp.status_code}")
            
        except Exception as e:
            print(f" ❌ {str(e)[:50]}")
        
        current_date += pd.Timedelta(days=1)
        time.sleep(2)
    
    if not all_new_rows:
        print(f"⚠️  No new data fetched")
        return df_cached
    
    df_new = pd.DataFrame(all_new_rows)
    print(f"📥 Total: {len(df_new)} new rows")
    
    if 'block_date' not in df_new.columns:
        print(f"❌ Missing block_date!")
        return df_cached
    
    df_new["block_date"] = pd.to_datetime(df_new["block_date"], utc=True)
    
    # Merge with cache
    if not df_cached.empty:
        common_cols = list(set(df_cached.columns) & set(df_new.columns))
        df_cached = df_cached[common_cols]
        df_new = df_new[common_cols]
        
        df_combined = pd.concat([df_cached, df_new], ignore_index=True)
        df_combined = df_combined.drop_duplicates(
            subset=['block_date'],
            keep='last'
        ).sort_values('block_date').reset_index(drop=True)
        
        print(f"📊 Combined: {len(df_combined)} rows")
    else:
        df_combined = df_new
        print(f"📊 New dataset: {len(df_combined)} rows")
    
    # Update cache
    update_cache_file(cache_path, df_combined)
    
    return df_combined
def update_cache_file(cache_path, df):
    try:
        if 'block_date' in df.columns:
            df_dates = df['block_date'].copy()
            
            df_serializable = df.copy()
            df_serializable['block_date'] = df_serializable['block_date'].dt.strftime('%Y-%m-%d')
            
            with open(cache_path, 'w') as f:
                json.dump({
                    "last_block_date": df_dates.max().strftime("%Y-%m-%d"),
                    "data": json.loads(df_serializable.to_json(orient="records", date_format='iso'))
                }, f, indent=2)
            
            print(f"💾 Cache updated to {df_dates.max().date()}")
            
    except Exception as e:
        print(f"❌ Cache update failed: {e}")


- Data Fetching for Bitcoin and Etherum from Coingecko

In [20]:

def fetch_cg_chunked(cg_id, start_date_str, end_date, key, days=30):
    url = "https://pro-api.coingecko.com/api/v3"
    headers = {"x-cg-pro-api-key": key}
    
    if isinstance(start_date_str, str):
        try:
            start_dt = pd.to_datetime(start_date_str, format='%d-%m-%Y', utc=True)
        except:
            start_dt = pd.to_datetime(start_date_str, utc=True)
    else:
        start_dt = to_utc(start_date_str)
    
    end_dt = to_utc(end_date) + pd.Timedelta(days=1)
    
    all_prices, curr = [], start_dt
    
    while curr < end_dt:
        next_dt = min(curr + pd.Timedelta(days=days), end_dt)
        params = {
            "vs_currency": "usd", 
            "from": int(curr.timestamp()), 
            "to": int(next_dt.timestamp())
        }
        
        try:
            r = requests.get(
                f"{url}/coins/{cg_id}/market_chart/range", 
                params=params, 
                headers=headers, 
                timeout=30
            )
            
            if r.status_code == 200:
                prices = r.json().get("prices", [])
                if prices:
                    all_prices.extend(prices)
            
        except Exception:
            pass
        
        time.sleep(0.5)
        curr = next_dt
    
    if not all_prices:
        return pd.DataFrame()
    
    df = pd.DataFrame(all_prices, columns=["timestamp", "price"])
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True).dt.floor("D")
    return df.groupby("date")["price"].mean().reset_index()

In [21]:
def get_price_incremental(symbol, cg_id, start_date_str, end, key, force_fetch=False):
    cache_path = f"data/price_cache/{symbol}.csv"
    today_utc = pd.Timestamp.utcnow().floor("D")
    yesterday = today_utc - pd.Timedelta(days=1)
    end_dt = min(to_utc(end), yesterday)
    
    print(f"\n💰 Fetching {symbol.upper()} prices...")
    
    if force_fetch and os.path.exists(cache_path):
        os.remove(cache_path)
    
    df_cached = pd.DataFrame()
    if os.path.exists(cache_path) and not force_fetch:
        try:
            df_cached = pd.read_csv(cache_path, parse_dates=["date"])
            df_cached["date"] = df_cached["date"].apply(to_utc)
            
            if not df_cached.empty:
                last_date = df_cached["date"].max()
                first_date = df_cached["date"].min()
                expected_start = to_utc(start_date_str)
                
                needs_historical = first_date > expected_start
                needs_updates = last_date < end_dt
                
                if not needs_historical and not needs_updates:
                    print(f"✅ {symbol.upper()} current ({first_date.date()} to {last_date.date()})")
                    return df_cached
                
                fetch_ranges = []
                
                if needs_historical:
                    fetch_ranges.append((expected_start, first_date - pd.Timedelta(days=1)))
                
                if needs_updates:
                    fetch_ranges.append((last_date + pd.Timedelta(days=1), end_dt))
                
                all_new_data = []
                for fetch_start, fetch_end in fetch_ranges:
                    if fetch_start <= fetch_end:
                        new_data = fetch_cg_chunked(cg_id, fetch_start, fetch_end, key)
                        if not new_data.empty:
                            all_new_data.append(new_data)
                
                if not all_new_data:
                    return df_cached
                
                df_new = pd.concat(all_new_data, ignore_index=True)
                df_new = df_new.rename(columns={"price": f"{symbol}_price"})
                
                df_combined = pd.concat([df_cached, df_new], ignore_index=True)
                df_combined = df_combined.drop_duplicates("date").sort_values("date").reset_index(drop=True)
                
                print(f"📊 {symbol.upper()}: {len(df_combined)} total")
                
        except Exception as e:
            print(f"⚠️  Cache error: {e}")
            df_cached = pd.DataFrame()
            fetch_start = to_utc(start_date_str)
    else:
        fetch_start = to_utc(start_date_str)
    
    if df_cached.empty:
        new_data = fetch_cg_chunked(cg_id, fetch_start, end_dt, key)
        
        if new_data.empty:
            return pd.DataFrame()
        
        df_combined = new_data.rename(columns={"price": f"{symbol}_price"})
    
    df_combined.to_csv(cache_path, index=False)
    print(f"💾 {symbol.upper()} saved: {df_combined['date'].min().date()} to {df_combined['date'].max().date()}")
    
    return df_combined


- Incrementally Loading and from dune and coingecko

In [22]:
def load_all_data_incremental(force_fetch=False):
    print("📊 Loading data...")
    
    df_whales = fetch_dune_incremental(
        QUERIES["whales"][0], 
        QUERIES["whales"][1],
        query_name="whale_data",
        force_fetch=force_fetch
    )
    df_whales.to_csv(QUERIES["whales"][2], index=False)
    
    time.sleep(2)
    
    df_market = fetch_dune_incremental(
        QUERIES["market_intent"][0], 
        QUERIES["market_intent"][1],
        query_name="market_intent",
        force_fetch=force_fetch
    )
    df_market.to_csv(QUERIES["market_intent"][2], index=False)
    
    max_date = max(
        df_whales["block_date"].max() if not df_whales.empty else DUNE_START,
        df_market["block_date"].max() if not df_market.empty else DUNE_START
    )
    
    df_btc = get_price_incremental("btc", "bitcoin", COINGECKO_BTC_START, max_date, COINGECKO_API_KEY, force_fetch)
    df_eth = get_price_incremental("eth", "ethereum", COINGECKO_ETH_START, max_date, COINGECKO_API_KEY, force_fetch)
    
    print(f"\n📈 Summary:")
    print(f"   Whale: {len(df_whales)} rows")
    print(f"   Market: {len(df_market)} rows")
    print(f"   BTC: {len(df_btc)} rows")
    print(f"   ETH: {len(df_eth)} rows")
    
    return df_whales, df_market, df_btc, df_eth



In [23]:
def load_cached_data():
    print("📂 Loading cached data...")
    
    files = {
        'whale': 'data/whale_ml_ready.csv',
        'market': 'data/market_intent_ml_ready.csv',
        'btc': 'data/price_cache/btc.csv',
        'eth': 'data/price_cache/eth.csv'
    }
    
    loaded = {}
    
    for name, path in files.items():
        if os.path.exists(path):
            try:
                if name in ['btc', 'eth']:
                    df = pd.read_csv(path, parse_dates=["date"])
                    df["date"] = df["date"].apply(to_utc)
                else:
                    df = pd.read_csv(path, parse_dates=["block_date"])
                    df["block_date"] = df["block_date"].apply(to_utc)
                
                loaded[name] = df
                print(f"✅ {name}: {len(df)} rows")
            except Exception as e:
                print(f"❌ {name}: {e}")
                loaded[name] = pd.DataFrame()
        else:
            print(f"⚠️  {name} not found")
            loaded[name] = pd.DataFrame()
    
    return (loaded.get('whale', pd.DataFrame()),
            loaded.get('market', pd.DataFrame()),
            loaded.get('btc', pd.DataFrame()),
            loaded.get('eth', pd.DataFrame()))

- Main Loading Execution

In [24]:

if __name__ == "__main__":
    print("\n" + "="*70)
    print("📊 DATA LOADER - FIXED VERSION")
    print("="*70)
    
    print("\n📋 OPTIONS:")
    print("1️⃣  Fetch fresh (incremental)")
    print("2️⃣  Force re-fetch all")
    print("3️⃣  Load cached only")
    print("="*70)

    choice = input("\nSelect (1-3): ").strip()

    if choice == '1':
        print("\n🚀 Fetching fresh data...")
        load_all_data_incremental(force_fetch=False)
        
    elif choice == '2':
        confirm = input("Delete cache and fetch all? (y/n): ").lower()
        if confirm == 'y':
            load_all_data_incremental(force_fetch=True)
        
    elif choice == '3':
        load_cached_data()
        
    else:
        print("❌ Invalid option")
    
    print(f"\n{'='*70}")
    print("✅ Complete!")
    print(f"{'='*70}")


📊 DATA LOADER - FIXED VERSION

📋 OPTIONS:
1️⃣  Fetch fresh (incremental)
2️⃣  Force re-fetch all
3️⃣  Load cached only

🚀 Fetching fresh data...
📊 Loading data...

📊 Fetching whale_data...
   Using 5 API keys
✅ whale_data cache current (2026-01-04)

📊 Fetching market_intent...
   Using 5 API keys
✅ market_intent cache current (2026-01-04)

💰 Fetching BTC prices...
📊 BTC: 4633 total
💾 BTC saved: 2013-04-28 to 2026-01-04

💰 Fetching ETH prices...
📊 ETH: 3803 total
💾 ETH saved: 2015-08-07 to 2026-01-04

📈 Summary:
   Whale: 3003 rows
   Market: 3003 rows
   BTC: 4633 rows
   ETH: 3803 rows

✅ Complete!


- Exploratory Data Analysis (EDA)

In [ ]:
pd.read_csv('data/whale_ml_ready.csv')

,whale_exchange_withdrawals_eth,deposit_tx_count,mega_whale_ratio,exchange_volume_ratio,net_flow_ma7,whale_tx_count,non_exchange_volume_eth,non_exchange_tx_count,std_whale_tx_size_eth,whale_net_exchange_flow_eth,whale_volume_eth,mega_whale_volume_eth,deposit_withdrawal_ratio,withdrawal_tx_count,whale_exchange_deposits_eth,mega_whale_tx_count,non_exchange_ratio,block_date
0,22147.0794,9.0,0.8971,0.0902,10749.2728,190.0,3.382636e+05,176.0,2310.607296,10749.2728,3.718084e+05,3.335630e+05,0.5146,5.0,11397.8066,149.0,0.9098,2017-10-16 00:00:00+00:00
1,39067.9609,21.0,0.9456,0.0885,-20684.9471,250.0,1.017314e+06,214.0,13716.203240,-20684.9471,1.116134e+06,1.055398e+06,1.5295,15.0,59752.9080,185.0,0.9115,2017-10-17 00:00:00+00:00
2,15570.6242,5.0,1.0000,0.0414,-15211.3463,349.0,9.463566e+05,339.0,3957.459236,-9737.7455,9.872356e+05,9.872356e+05,1.6254,5.0,25308.3697,349.0,0.9586,2017-10-18 00:00:00+00:00
3,49041.2315,6.0,0.7803,0.1034,3912.9137,336.0,4.850454e+05,324.0,2819.239036,42161.4337,5.409664e+05,4.221430e+05,0.1403,6.0,6879.7978,217.0,0.8966,2017-10-19 00:00:00+00:00
4,20589.5335,11.0,0.8360,0.0619,3137.9499,319.0,6.113884e+05,298.0,3236.014159,813.0587,6.517544e+05,5.448516e+05,0.9605,10.0,19776.4748,212.0,0.9381,2017-10-20 00:00:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2998,78169.1800,23.0,1.0000,0.3820,-71795.0952,74.0,4.704587e+05,45.0,5799.701052,-134447.6876,7.612448e+05,7.612448e+05,2.7200,6.0,212616.8676,74.0,0.6180,2025-12-31 00:00:00+00:00
2999,15451.6737,4.0,1.0000,0.1213,-66926.8169,44.0,2.381382e+05,38.0,4751.567011,-1976.0069,2.710176e+05,2.710176e+05,1.1279,2.0,17427.6806,44.0,0.8787,2026-01-01 00:00:00+00:00
3000,94093.4141,18.0,1.0000,0.4098,-38461.2988,70.0,3.617799e+05,45.0,4361.441154,-63049.3965,6.130161e+05,6.130161e+05,1.6701,7.0,157142.8106,70.0,0.5902,2026-01-02 00:00:00+00:00
3001,15084.6473,8.0,1.0000,0.1561,-32946.0600,45.0,3.412452e+05,35.0,12264.236700,-32946.0600,4.043606e+05,4.043606e+05,3.1841,2.0,48030.7073,45.0,0.8439,2026-01-03 00:00:00+00:00


In [28]:
pd.read_csv('data/whale_ml_ready.csv').dtypes

whale_exchange_withdrawals_eth    float64
deposit_tx_count                  float64
mega_whale_ratio                  float64
exchange_volume_ratio             float64
net_flow_ma7                      float64
whale_tx_count                    float64
non_exchange_volume_eth           float64
non_exchange_tx_count             float64
std_whale_tx_size_eth             float64
whale_net_exchange_flow_eth       float64
whale_volume_eth                  float64
mega_whale_volume_eth             float64
deposit_withdrawal_ratio          float64
withdrawal_tx_count               float64
whale_exchange_deposits_eth       float64
mega_whale_tx_count               float64
non_exchange_ratio                float64
block_date                         object
dtype: object

In [35]:
pd.read_csv('data/whale_ml_ready.csv').isnull().sum()

whale_exchange_withdrawals_eth    0
deposit_tx_count                  0
mega_whale_ratio                  0
exchange_volume_ratio             0
net_flow_ma7                      0
whale_tx_count                    0
non_exchange_volume_eth           0
non_exchange_tx_count             0
std_whale_tx_size_eth             0
whale_net_exchange_flow_eth       0
whale_volume_eth                  0
mega_whale_volume_eth             0
deposit_withdrawal_ratio          0
withdrawal_tx_count               0
whale_exchange_deposits_eth       0
mega_whale_tx_count               0
non_exchange_ratio                0
block_date                        0
dtype: int64

In [30]:
pd.read_csv('data/market_intent_ml_ready.csv')

,exchange_flow_share,tx_per_active_zscore_90d,smart_contract_ratio_delta_1d,net_exchange_flow_ratio,tx_per_active_delta_1d,whale_exchange_asymmetry,whale_exchange_flow_ratio,eth_burned_zscore_90d,whale_volume_ratio_delta_1d,median_gas_delta_7d,block_date,whale_tx_zscore_90d,block_fullness_delta_1d,eth_burned_delta_1d,whale_volume_ratio,whale_volume_ratio_delta_3d,median_gas_delta_1d
0,0.015021,0.0000,NaN,0.002401,NaN,0.274629,0.001739,0.0000,NaN,NaN,2017-10-16 00:00:00+00:00,0.0000,NaN,NaN,0.237421,NaN,NaN
1,0.015033,-0.7071,0.000083,-0.000963,-0.0661,-0.187957,-0.001727,0.0000,0.027758,NaN,2017-10-17 00:00:00+00:00,0.7071,-0.145119,NaN,0.265178,NaN,-10.3854
2,0.010296,0.2477,0.024143,-0.000241,0.0456,-0.250389,-0.000781,0.0000,-0.065376,NaN,2017-10-18 00:00:00+00:00,0.0901,-0.057348,NaN,0.199802,NaN,3.2781
3,0.008123,0.9788,0.043882,0.003385,0.0392,0.753946,0.003410,0.0000,-0.005306,NaN,2017-10-19 00:00:00+00:00,0.2059,0.058294,NaN,0.194496,-0.042924,-0.1501
4,0.007958,1.1727,-0.022849,0.001450,0.0256,0.020142,0.000072,0.0000,0.027682,NaN,2017-10-20 00:00:00+00:00,0.2428,0.062177,NaN,0.222178,-0.043000,-3.0014
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2998,0.329380,-1.1968,0.010461,-0.063486,0.1617,-0.189408,-0.061575,-0.2551,0.006488,0.0660,2025-12-31 00:00:00+00:00,1.1318,0.000579,0.7059,0.950323,0.059050,0.0136
2999,0.144672,-1.2931,-0.051376,-0.026753,-0.0168,-0.168902,-0.023435,-0.2668,-0.038538,0.0185,2026-01-01 00:00:00+00:00,-1.4677,0.002226,-3.9598,0.911785,-0.030471,-0.0457
3000,0.383195,-0.7325,0.036647,-0.065930,0.0812,-0.167513,-0.063541,-0.2337,0.036573,0.0586,2026-01-02 00:00:00+00:00,0.8464,-0.001872,9.0262,0.948359,0.004523,0.0482
3001,0.211788,0.3186,-0.030336,-0.028868,0.1529,-0.124502,-0.025588,-0.2608,-0.030761,0.0270,2026-01-03 00:00:00+00:00,-1.4696,0.000748,-7.5086,0.917597,-0.032726,-0.0477


In [34]:
pd.read_csv('data/market_intent_ml_ready.csv').dtypes

exchange_flow_share              float64
tx_per_active_zscore_90d         float64
smart_contract_ratio_delta_1d    float64
net_exchange_flow_ratio          float64
tx_per_active_delta_1d           float64
whale_exchange_asymmetry         float64
whale_exchange_flow_ratio        float64
eth_burned_zscore_90d            float64
whale_volume_ratio_delta_1d      float64
median_gas_delta_7d              float64
block_date                        object
whale_tx_zscore_90d              float64
block_fullness_delta_1d          float64
eth_burned_delta_1d              float64
whale_volume_ratio               float64
whale_volume_ratio_delta_3d      float64
median_gas_delta_1d              float64
dtype: object

In [36]:
pd.read_csv('data/market_intent_ml_ready.csv').isnull().sum()

exchange_flow_share                 0
tx_per_active_zscore_90d            0
smart_contract_ratio_delta_1d       1
net_exchange_flow_ratio             0
tx_per_active_delta_1d              1
whale_exchange_asymmetry            0
whale_exchange_flow_ratio           0
eth_burned_zscore_90d               0
whale_volume_ratio_delta_1d         1
median_gas_delta_7d                 7
block_date                          0
whale_tx_zscore_90d                 0
block_fullness_delta_1d             1
eth_burned_delta_1d              1390
whale_volume_ratio                  0
whale_volume_ratio_delta_3d         3
median_gas_delta_1d                 1
dtype: int64